# Multi-Document Summarization Test

This notebook tests multiple multi-document summarization methods on:
1. **Topic groups**: Articles belonging to the same BERTopic-discovered semantic category
2. **Event clusters**: Articles covering the same news event across multiple sources

Methods tested:
- **PRIMERA** (`allenai/primera`) - Designed for multi-doc summarization
- **LED** (`allenai/led-base-16384`) - Long-context transformer
- **LongT5** (`google/long-t5-tglobal-base`) - Long-context T5
- **OpenAI GPT-4o** - LLM-based
- **Gemini** (2.0 Flash) - LLM-based

## Section 1: Setup & Data Loading

In [ ]:
# Cell 1: Imports
import sys
sys.path.append('..')

from src.db import Database
from src.config import load_config
from src.multi_doc_summarization import (
    PRIMERASummarizer,
    LEDMultiDocSummarizer,
    LongT5MultiDocSummarizer,
    OpenAIMultiDocSummarizer,
    GeminiMultiDocSummarizer
)
from dashboard.components.source_mapping import SOURCE_NAMES

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports complete")

In [ ]:
# Cell 2: Cache configuration
import json
import os
from pathlib import Path
from datetime import datetime

# Cache directory for storing results (separate file per model)
CACHE_DIR = Path("cache/multi_doc_summarization")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHE = True  # Set to False to regenerate all summaries

# Methods to test
METHODS = ['primera', 'led', 'longt5', 'openai', 'gemini']

def get_cache_path(method: str, group_type: str) -> Path:
    """Get cache file path for method and group type (topics/clusters)."""
    return CACHE_DIR / f"{method}_{group_type}.json"

def load_cache(method: str, group_type: str) -> dict:
    """Load cached summaries for a specific method if available."""
    cache_path = get_cache_path(method, group_type)
    if not USE_CACHE or not cache_path.exists():
        return None
    
    with open(cache_path, 'r') as f:
        data = json.load(f)
    print(f"  ✓ Loaded {method} {group_type} from cache ({cache_path.name})")
    return data

def save_cache(method: str, group_type: str, results: list, 
               version_id: str, version_name: str, groups_tested: list):
    """Save summaries for a specific method to cache with metadata."""
    cache_path = get_cache_path(method, group_type)
    
    cache_data = {
        "metadata": {
            "method": method,
            "group_type": group_type,
            "version_id": version_id,
            "version_name": version_name,
            "groups_tested": groups_tested,
            "created_at": datetime.now().isoformat(),
            "result_count": len(results)
        },
        "results": results
    }
    
    with open(cache_path, 'w') as f:
        json.dump(cache_data, f, indent=2, default=str)
    print(f"  ✓ Saved {method} {group_type} to cache ({cache_path.name})")

def load_all_topic_results() -> dict:
    """Load all cached topic results."""
    results = {}
    for method in METHODS:
        cache_data = load_cache(method, 'topics')
        if cache_data:
            results[method] = cache_data['results']
    return results if len(results) == len(METHODS) else None

def load_all_cluster_results() -> dict:
    """Load all cached cluster results."""
    results = {}
    for method in METHODS:
        cache_data = load_cache(method, 'clusters')
        if cache_data:
            results[method] = cache_data['results']
    return results if len(results) == len(METHODS) else None

def check_cache_status():
    """Check which cache files exist."""
    print("Cache status:")
    for group_type in ['topics', 'clusters']:
        print(f"\n  {group_type.capitalize()}:")
        for method in METHODS:
            cache_path = get_cache_path(method, group_type)
            status = "✓" if cache_path.exists() else "✗"
            print(f"    {status} {method}")

print(f"Cache directory: {CACHE_DIR}")
print(f"Cache mode: {'ENABLED' if USE_CACHE else 'DISABLED'}\n")
check_cache_status()

In [ ]:
# Cell 2: Verify API keys

from src.config import load_config
import os

config = load_config()

openai_key = config.get('openai', {}).get('api_key') or os.environ.get('OPENAI_API_KEY')
if openai_key and openai_key != 'YOUR_OPENAI_API_KEY_HERE':
    print('✓ OPENAI_API_KEY is configured')
else:
    print('⚠️  OPENAI_API_KEY not set (OpenAI cells will fail)')

gemini_key = config.get('gemini', {}).get('api_key') or os.environ.get('GOOGLE_API_KEY')
if gemini_key and gemini_key != 'YOUR_GOOGLE_API_KEY_HERE':
    print('✓ GOOGLE_API_KEY is configured')
else:
    print('⚠️  GOOGLE_API_KEY not set (Gemini cells will fail)')

In [ ]:
# Cell 3: Load config and connect to database
config = load_config()
db = Database()
db.connect()

print("✓ Database connected")
print(f"Schema: {db.config['schema']}")

In [ ]:
# Cell 4: Select versions for testing
# Get completed topic versions
schema = db.config['schema']
with db.cursor() as cur:
    cur.execute(f"""
        SELECT id, name, description, configuration, pipeline_status
        FROM {schema}.result_versions
        WHERE analysis_type = 'topics'
          AND (pipeline_status->>'topics')::boolean = true
          AND configuration->'embeddings'->>'model' = 'google/embeddinggemma-300m'
        ORDER BY created_at DESC
        LIMIT 1
    """)
    topic_version = cur.fetchone()

if not topic_version:
    raise ValueError("No completed topic version found with google/embeddinggemma-300m")

topic_version_id = str(topic_version['id'])
print(f"Selected topic version: {topic_version['name']}")
print(f"  ID: {topic_version_id}")
print(f"  Description: {topic_version['description']}")

# Get completed clustering versions
with db.cursor() as cur:
    cur.execute(f"""
        SELECT id, name, description, configuration, pipeline_status
        FROM {schema}.result_versions
        WHERE analysis_type = 'clustering'
          AND (pipeline_status->>'clustering')::boolean = true
        ORDER BY created_at DESC
        LIMIT 1
    """)
    clustering_version = cur.fetchone()

if not clustering_version:
    raise ValueError("No completed clustering version found")

clustering_version_id = str(clustering_version['id'])
print(f"\nSelected clustering version: {clustering_version['name']}")
print(f"  ID: {clustering_version_id}")
print(f"  Description: {clustering_version['description']}")

In [ ]:
# Cell 5: Load sample topics (10-20 articles per topic)
topics = db.get_all_topics_with_counts(topic_version_id, min_article_count=10)
topics_df = pd.DataFrame(topics)

# Filter to topics with 10-20 articles (not too small, not too large)
topics_df = topics_df[(topics_df['article_count'] >= 10) & (topics_df['article_count'] <= 20)]

print(f"Found {len(topics_df)} topics with 10-20 articles")
print("\nTop 10 topics by article count:")
display(topics_df.head(10)[['name', 'article_count', 'keywords']])

# Select 5 diverse topics for testing
selected_topics = topics_df.head(5)['id'].tolist()
print(f"\nSelected {len(selected_topics)} topics for testing")

In [ ]:
# Cell 6: Load sample event clusters (3-10 articles per cluster)
clusters = db.get_all_clusters_with_counts(clustering_version_id, min_article_count=3)
clusters_df = pd.DataFrame(clusters)

# Filter to clusters with 3-10 articles
clusters_df = clusters_df[(clusters_df['article_count'] >= 3) & (clusters_df['article_count'] <= 10)]

print(f"Found {len(clusters_df)} clusters with 3-10 articles")
print("\nTop 10 clusters by article count:")
display(clusters_df.head(10)[['cluster_name', 'article_count', 'sources_count', 'date_start', 'date_end']])

# Select 5 diverse clusters for testing
selected_clusters = clusters_df.head(5)['id'].tolist()
print(f"\nSelected {len(selected_clusters)} clusters for testing")

In [ ]:
# Cell 7: Fetch articles for selected groups
# Store topic groups
topic_groups = {}
for topic_id in selected_topics:
    articles = db.get_articles_by_topic(topic_id, topic_version_id)
    topic_name = topics_df[topics_df['id'] == topic_id].iloc[0]['name']
    topic_groups[topic_name] = articles
    print(f"Topic '{topic_name}': {len(articles)} articles from {len(set(a['source_id'] for a in articles))} sources")

print()

# Store cluster groups
cluster_groups = {}
for cluster_id in selected_clusters:
    articles = db.get_articles_by_cluster(cluster_id, clustering_version_id)
    cluster_name = clusters_df[clusters_df['id'] == cluster_id].iloc[0]['cluster_name']
    cluster_groups[cluster_name] = articles
    print(f"Cluster '{cluster_name}': {len(articles)} articles from {len(set(a['source_id'] for a in articles))} sources")

## Section 2: Method Comparison on Topics

In [ ]:
# Cell: Load cached topic results or prepare for fresh generation
print("Loading topic results from cache...\n")

# Try to load cached results for each method
primera_topic_results = None
led_topic_results = None
longt5_topic_results = None
openai_topic_results = None
gemini_topic_results = None

if USE_CACHE:
    primera_cache = load_cache('primera', 'topics')
    led_cache = load_cache('led', 'topics')
    longt5_cache = load_cache('longt5', 'topics')
    openai_cache = load_cache('openai', 'topics')
    gemini_cache = load_cache('gemini', 'topics')
    
    if primera_cache:
        primera_topic_results = primera_cache['results']
    if led_cache:
        led_topic_results = led_cache['results']
    if longt5_cache:
        longt5_topic_results = longt5_cache['results']
    if openai_cache:
        openai_topic_results = openai_cache['results']
    if gemini_cache:
        gemini_topic_results = gemini_cache['results']

# Summary
cached = [m for m, r in [('PRIMERA', primera_topic_results), ('LED', led_topic_results), 
                          ('LongT5', longt5_topic_results), ('OpenAI', openai_topic_results),
                          ('Gemini', gemini_topic_results)] if r is not None]
missing = [m for m, r in [('PRIMERA', primera_topic_results), ('LED', led_topic_results), 
                           ('LongT5', longt5_topic_results), ('OpenAI', openai_topic_results),
                           ('Gemini', gemini_topic_results)] if r is None]

print(f"\n✓ Cached: {', '.join(cached) if cached else 'None'}")
print(f"✗ Missing (will generate): {', '.join(missing) if missing else 'None'}")

In [ ]:
# Cell 8: Test PRIMERA on topics
if primera_topic_results is not None:
    print("✓ PRIMERA topic results already loaded from cache - skipping generation")
else:
    print("Testing PRIMERA on topic groups...\n")

    # Initialize PRIMERA summarizer
    primera_config = {
        'method': 'primera'
    }
    primera = PRIMERASummarizer(primera_config)

    primera_topic_results = []

    for topic_name, articles in topic_groups.items():
        print(f"\n{'='*80}")
        print(f"Topic: {topic_name}")
        print(f"Articles: {len(articles)}")
        print(f"{'='*80}")
        
        # Extract documents and sources
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        # Show article titles
        print("\nArticles in group:")
        for i, article in enumerate(articles):
            source_name = SOURCE_NAMES.get(article['source_id'], article['source_id'])
            print(f"  {i+1}. [{source_name}] {article['title'][:70]}...")
        
        # Generate summary
        start_time = time.time()
        summary = primera.summarize_multiple(documents, sources)
        processing_time = time.time() - start_time
        
        print(f"\nPRIMERA Summary ({processing_time:.2f}s):")
        print(summary)
        
        primera_topic_results.append({
            'group': topic_name,
            'article_count': len(articles),
            'summary': summary,
            'processing_time': processing_time,
            'word_count': primera.count_words(summary)
        })

    avg_time = sum(r['processing_time'] for r in primera_topic_results) / len(primera_topic_results)
    print(f"\n\nAverage processing time: {avg_time:.2f}s")
    
    # Save immediately after generation
    save_cache('primera', 'topics', primera_topic_results, 
               topic_version_id, topic_version['name'], list(topic_groups.keys()))

In [ ]:
# Cell 9: Test LED on topics
if led_topic_results is not None:
    print("✓ LED topic results already loaded from cache - skipping generation")
else:
    print("Testing LED on topic groups...\n")

    # Initialize LED summarizer
    led_config = {
        'method': 'led'
    }
    led = LEDMultiDocSummarizer(led_config)

    led_topic_results = []

    for topic_name, articles in topic_groups.items():
        print(f"\n{'='*80}")
        print(f"Topic: {topic_name}")
        print(f"{'='*80}")
        
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        start_time = time.time()
        summary = led.summarize_multiple(documents, sources)
        processing_time = time.time() - start_time
        
        print(f"LED Summary ({processing_time:.2f}s):")
        print(summary)
        
        led_topic_results.append({
            'group': topic_name,
            'article_count': len(articles),
            'summary': summary,
            'processing_time': processing_time,
            'word_count': led.count_words(summary)
        })

    avg_time = sum(r['processing_time'] for r in led_topic_results) / len(led_topic_results)
    print(f"\n\nAverage processing time: {avg_time:.2f}s")
    
    # Save immediately after generation
    save_cache('led', 'topics', led_topic_results, 
               topic_version_id, topic_version['name'], list(topic_groups.keys()))

In [ ]:
# Cell 10: Test LongT5 on topics
if longt5_topic_results is not None:
    print("✓ LongT5 topic results already loaded from cache - skipping generation")
else:
    print("Testing LongT5 on topic groups...\n")

    # Initialize LongT5 summarizer
    longt5_config = {
        'method': 'longt5'
    }
    longt5 = LongT5MultiDocSummarizer(longt5_config)

    longt5_topic_results = []

    for topic_name, articles in topic_groups.items():
        print(f"\n{'='*80}")
        print(f"Topic: {topic_name}")
        print(f"{'='*80}")
        
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        start_time = time.time()
        summary = longt5.summarize_multiple(documents, sources)
        processing_time = time.time() - start_time
        
        print(f"LongT5 Summary ({processing_time:.2f}s):")
        print(summary)
        
        longt5_topic_results.append({
            'group': topic_name,
            'article_count': len(articles),
            'summary': summary,
            'processing_time': processing_time,
            'word_count': longt5.count_words(summary)
        })

    avg_time = sum(r['processing_time'] for r in longt5_topic_results) / len(longt5_topic_results)
    print(f"\n\nAverage processing time: {avg_time:.2f}s")
    
    # Save immediately after generation
    save_cache('longt5', 'topics', longt5_topic_results, 
               topic_version_id, topic_version['name'], list(topic_groups.keys()))

In [ ]:
# Cell 11: Test OpenAI on topics
if openai_topic_results is not None:
    print("✓ OpenAI topic results already loaded from cache - skipping generation")
else:
    print("Testing OpenAI GPT-4o on topic groups...\n")

    # Initialize OpenAI summarizer
    openai_config = {
        'method': 'openai',
        'llm_model': 'gpt-4o',
        'llm_temperature': 0.0
    }
    openai = OpenAIMultiDocSummarizer(openai_config)

    openai_topic_results = []

    for topic_name, articles in topic_groups.items():
        print(f"\n{'='*80}")
        print(f"Topic: {topic_name}")
        print(f"{'='*80}")
        
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        start_time = time.time()
        summary = openai.summarize_multiple(documents, sources)
        processing_time = time.time() - start_time
        
        print(f"OpenAI Summary ({processing_time:.2f}s):")
        print(summary)
        
        openai_topic_results.append({
            'group': topic_name,
            'article_count': len(articles),
            'summary': summary,
            'processing_time': processing_time,
            'word_count': openai.count_words(summary)
        })

    avg_time = sum(r['processing_time'] for r in openai_topic_results) / len(openai_topic_results)
    print(f"\n\nAverage processing time: {avg_time:.2f}s")
    
    # Save immediately after generation
    save_cache('openai', 'topics', openai_topic_results, 
               topic_version_id, topic_version['name'], list(topic_groups.keys()))

In [ ]:
# Cell 12: Test Gemini on topics
if gemini_topic_results is not None:
    print("✓ Gemini topic results already loaded from cache - skipping generation")
else:
    print("Testing Gemini on topic groups...\n")

    # Initialize Gemini summarizer
    gemini_config = {
        'method': 'gemini',
        'llm_model': 'gemini-2.0-flash',
        'llm_temperature': 0.0
    }
    gemini = GeminiMultiDocSummarizer(gemini_config)

    gemini_topic_results = []

    for topic_name, articles in topic_groups.items():
        print(f"\n{'='*80}")
        print(f"Topic: {topic_name}")
        print(f"{'='*80}")
        
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        start_time = time.time()
        summary = gemini.summarize_multiple(documents, sources)
        processing_time = time.time() - start_time
        
        print(f"Gemini Summary ({processing_time:.2f}s):")
        print(summary)
        
        gemini_topic_results.append({
            'group': topic_name,
            'article_count': len(articles),
            'summary': summary,
            'processing_time': processing_time,
            'word_count': gemini.count_words(summary)
        })

    avg_time = sum(r['processing_time'] for r in gemini_topic_results) / len(gemini_topic_results)
    print(f"\n\nAverage processing time: {avg_time:.2f}s")
    
    # Save immediately after generation
    save_cache('gemini', 'topics', gemini_topic_results, 
               topic_version_id, topic_version['name'], list(topic_groups.keys()))

In [ ]:
# Cell 13: Compare topic summaries
print("\nTopic Summaries Comparison\n" + "="*80 + "\n")

# Create comparison dataframe
comparison_data = []
for method_name, results in [
    ('PRIMERA', primera_topic_results),
    ('LED', led_topic_results),
    ('LongT5', longt5_topic_results),
    ('OpenAI', openai_topic_results),
    ('Gemini', gemini_topic_results)
]:
    for result in results:
        comparison_data.append({
            'Method': method_name,
            'Group': result['group'][:30] + '...' if len(result['group']) > 30 else result['group'],
            'Time (s)': f"{result['processing_time']:.2f}",
            'Words': result['word_count'],
            'Articles': result['article_count']
        })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Summary statistics
print("\n" + "="*80)
print("Summary Statistics")
print("="*80)
for method in ['PRIMERA', 'LED', 'LongT5', 'OpenAI', 'Gemini']:
    method_data = comparison_df[comparison_df['Method'] == method]
    avg_time = method_data['Time (s)'].astype(float).mean()
    avg_words = method_data['Words'].mean()
    print(f"{method:12s} - Avg time: {avg_time:5.2f}s, Avg words: {avg_words:6.1f}")

## Section 3: Method Comparison on Event Clusters

In [ ]:
# Cell: Load cached cluster results or prepare for fresh generation
print("Loading cluster results from cache...\n")

# Initialize cluster results dictionary
cluster_results_by_method = {
    'PRIMERA': None,
    'LED': None,
    'LongT5': None,
    'OpenAI': None,
    'Gemini': None
}

if USE_CACHE:
    primera_cache = load_cache('primera', 'clusters')
    led_cache = load_cache('led', 'clusters')
    longt5_cache = load_cache('longt5', 'clusters')
    openai_cache = load_cache('openai', 'clusters')
    gemini_cache = load_cache('gemini', 'clusters')
    
    if primera_cache:
        cluster_results_by_method['PRIMERA'] = primera_cache['results']
    if led_cache:
        cluster_results_by_method['LED'] = led_cache['results']
    if longt5_cache:
        cluster_results_by_method['LongT5'] = longt5_cache['results']
    if openai_cache:
        cluster_results_by_method['OpenAI'] = openai_cache['results']
    if gemini_cache:
        cluster_results_by_method['Gemini'] = gemini_cache['results']

# Summary
cached = [m for m, r in cluster_results_by_method.items() if r is not None]
missing = [m for m, r in cluster_results_by_method.items() if r is None]

print(f"\n✓ Cached: {', '.join(cached) if cached else 'None'}")
print(f"✗ Missing (will generate): {', '.join(missing) if missing else 'None'}")

In [ ]:
# Cell 14: Test all methods on event clusters

# Check which methods need generation
methods_to_run = [m for m, r in cluster_results_by_method.items() if r is None]

if not methods_to_run:
    print("✓ All cluster results already loaded from cache - skipping generation")
else:
    print(f"Generating cluster summaries for: {', '.join(methods_to_run)}\n")
    
    # Initialize summarizers only for methods that need generation
    summarizers = {}
    if 'PRIMERA' in methods_to_run:
        summarizers['PRIMERA'] = PRIMERASummarizer({'method': 'primera'})
        cluster_results_by_method['PRIMERA'] = []
    if 'LED' in methods_to_run:
        summarizers['LED'] = LEDMultiDocSummarizer({'method': 'led'})
        cluster_results_by_method['LED'] = []
    if 'LongT5' in methods_to_run:
        summarizers['LongT5'] = LongT5MultiDocSummarizer({'method': 'longt5'})
        cluster_results_by_method['LongT5'] = []
    if 'OpenAI' in methods_to_run:
        summarizers['OpenAI'] = OpenAIMultiDocSummarizer({'method': 'openai', 'llm_model': 'gpt-4o', 'llm_temperature': 0.0})
        cluster_results_by_method['OpenAI'] = []
    if 'Gemini' in methods_to_run:
        summarizers['Gemini'] = GeminiMultiDocSummarizer({'method': 'gemini', 'llm_model': 'gemini-2.0-flash', 'llm_temperature': 0.0})
        cluster_results_by_method['Gemini'] = []

    for cluster_name, articles in cluster_groups.items():
        print(f"\n{'='*80}")
        print(f"Cluster: {cluster_name}")
        print(f"Articles: {len(articles)} from {len(set(a['source_id'] for a in articles))} sources")
        print(f"{'='*80}\n")
        
        # Show article titles
        print("Articles in cluster:")
        for i, article in enumerate(articles):
            source_name = SOURCE_NAMES.get(article['source_id'], article['source_id'])
            print(f"  {i+1}. [{source_name}] {article['title'][:70]}...")
        print()
        
        documents = [a['content'] for a in articles]
        sources = [SOURCE_NAMES.get(a['source_id'], a['source_id']) for a in articles]
        
        # Test each method that needs generation
        for method_name, summarizer in summarizers.items():
            print(f"{method_name}:")
            start_time = time.time()
            summary = summarizer.summarize_multiple(documents, sources)
            processing_time = time.time() - start_time
            
            print(f"  ({processing_time:.2f}s) {summary}")
            print()
            
            cluster_results_by_method[method_name].append({
                'group': cluster_name,
                'article_count': len(articles),
                'summary': summary,
                'processing_time': processing_time,
                'word_count': summarizer.count_words(summary)
            })

    # Save results for each newly generated method
    method_file_map = {'PRIMERA': 'primera', 'LED': 'led', 'LongT5': 'longt5', 'OpenAI': 'openai', 'Gemini': 'gemini'}
    groups_tested = list(cluster_groups.keys())
    
    print("\nSaving cluster results...")
    for method_name in methods_to_run:
        save_cache(method_file_map[method_name], 'clusters', cluster_results_by_method[method_name],
                   clustering_version_id, clustering_version['name'], groups_tested)

    print("\n" + "="*80)
    print("Cluster testing complete")
    print("="*80)

In [ ]:
# Cell 15: Compare cluster summaries
print("\nEvent Cluster Summaries Comparison\n" + "="*80 + "\n")

# Create comparison dataframe
cluster_comparison_data = []
for method_name, results in cluster_results_by_method.items():
    for result in results:
        cluster_comparison_data.append({
            'Method': method_name,
            'Cluster': result['group'][:30] + '...' if len(result['group']) > 30 else result['group'],
            'Time (s)': f"{result['processing_time']:.2f}",
            'Words': result['word_count'],
            'Articles': result['article_count']
        })

cluster_comparison_df = pd.DataFrame(cluster_comparison_data)
display(cluster_comparison_df)

# Summary statistics
print("\n" + "="*80)
print("Summary Statistics (Event Clusters)")
print("="*80)
for method in ['PRIMERA', 'LED', 'LongT5', 'OpenAI', 'Gemini']:
    method_data = cluster_comparison_df[cluster_comparison_df['Method'] == method]
    avg_time = method_data['Time (s)'].astype(float).mean()
    avg_words = method_data['Words'].mean()
    print(f"{method:12s} - Avg time: {avg_time:5.2f}s, Avg words: {avg_words:6.1f}")

## Section 4: Performance Analysis

In [ ]:
# Cell 16: Processing time comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Topics
topic_times = {}
for method, results in [
    ('PRIMERA', primera_topic_results),
    ('LED', led_topic_results),
    ('LongT5', longt5_topic_results),
    ('OpenAI', openai_topic_results),
    ('Gemini', gemini_topic_results)
]:
    topic_times[method] = [r['processing_time'] for r in results]

axes[0].bar(topic_times.keys(), [sum(times)/len(times) for times in topic_times.values()])
axes[0].set_title('Average Processing Time - Topics')
axes[0].set_ylabel('Time (seconds)')
axes[0].tick_params(axis='x', rotation=45)

# Clusters
cluster_times = {}
for method, results in cluster_results_by_method.items():
    cluster_times[method] = [r['processing_time'] for r in results]

axes[1].bar(cluster_times.keys(), [sum(times)/len(times) for times in cluster_times.values()])
axes[1].set_title('Average Processing Time - Event Clusters')
axes[1].set_ylabel('Time (seconds)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 18: Summary statistics
# Word count distribution per method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Topics
topic_word_counts = {}
for method, results in [
    ('PRIMERA', primera_topic_results),
    ('LED', led_topic_results),
    ('LongT5', longt5_topic_results),
    ('OpenAI', openai_topic_results),
    ('Gemini', gemini_topic_results)
]:
    topic_word_counts[method] = [r['word_count'] for r in results]

axes[0].bar(topic_word_counts.keys(), [sum(counts)/len(counts) for counts in topic_word_counts.values()])
axes[0].set_title('Average Summary Length - Topics')
axes[0].set_ylabel('Word Count')
axes[0].tick_params(axis='x', rotation=45)

# Clusters
cluster_word_counts = {}
for method, results in cluster_results_by_method.items():
    cluster_word_counts[method] = [r['word_count'] for r in results]

axes[1].bar(cluster_word_counts.keys(), [sum(counts)/len(counts) for counts in cluster_word_counts.values()])
axes[1].set_title('Average Summary Length - Event Clusters')
axes[1].set_ylabel('Word Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 21: Close database connection
db.close()
print("✓ Database connection closed")
print("\nNotebook complete!")